# CinePal — GPU Embedding on Colab (stage 2)

Stage-2 of the ingestion pipeline. Downloads the cleaned TMDB snapshot pinned in
`configs/default.yaml` under `ingestion.artifacts.snapshot`, splits it into
main/mini/eval, embeds on a free Colab GPU, and publishes three timestamped
parquet files under `embeddings/` in the HF dataset repo.

Stage 1 (the actual TMDB scrape) runs **locally** via:

```bash
python -m dataset.scraper --upload   # produces snapshots/snapshot_YYYYMMDD.parquet
```

Paste the printed snapshot path into `configs/default.yaml` before opening
this notebook. Local ingest then loads the stage-2 outputs with no further
changes:

```bash
python -m db.ingest             # mini
python -m db.ingest --set main  # full set
```

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Click the 🔑 *Secrets* icon (left sidebar) and add:
   - `HF_TOKEN` — huggingface.co → Settings → Access Tokens (write scope)
   - `GITHUB_TOKEN` *(only if the repo is private)* — a PAT with `repo` scope
3. Fill in `REPO_URL` in cell 3.
4. Mount Google Drive (cell 5) — trailer embedding shards are persisted there
   across Colab sessions so a run can resume after bot detection interrupts it.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 1 — Verify GPU is available
!nvidia-smi

In [ ]:
# 2 — Install project dependencies
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q sentence-transformers pyarrow pandas numpy huggingface_hub pyyaml python-dotenv tqdm psycopg[binary] pgvector annotated-types pydantic pydantic-settings open-clip-torch Pillow
!pip install -q -U yt-dlp

In [ ]:
# 3 — Clone the repo and add it to the Python path
import os
import sys
from google.colab import userdata

REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/<REPO_NAME>.git"  # <-- fill in before running

# For private repos, inject the GitHub token into the clone URL.
try:
    github_token = userdata.get("GITHUB_TOKEN")
    if github_token:
        REPO_URL = REPO_URL.replace("https://", f"https://{github_token}@")
except Exception:
    pass  # secret not set — assume public repo

if not os.path.isdir("cantucci"):
    os.system(f"git clone {REPO_URL} cantucci")
else:
    os.system("git -C cantucci pull --ff-only")

os.chdir("cantucci")
if "." not in sys.path:
    sys.path.insert(0, ".")

print("Working directory:", os.getcwd())

In [ ]:
# 4 — Surface the required secret as an env var so backend.settings can read it
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError("Missing Colab secret: HF_TOKEN") from exc
print("Secrets loaded.")

In [ ]:
# 5 — Mount Google Drive for persistent trailer-embedding shard storage
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
SHARD_ROOT = Path("/content/drive/MyDrive/cinepal_trailer_shards")
print(f"Shard root: {SHARD_ROOT}")

In [ ]:
# 6 — Load project settings from configs/default.yaml
from backend.settings import get_settings, ARTIFACTS_DIR

cfg = get_settings()
print(
    f"Config loaded — embedding model: {cfg.representation.model}, "
    f"dim: {cfg.representation.embedding_dim}, split seed: {cfg.split.seed}, "
    f"mini_size: {cfg.split.mini_size}"
)

In [ ]:
# 7 — Download the cleaned snapshot from HF, then split
import pandas as pd
from dataset.hub.fetch import fetch_artifact
from dataset.transform import split

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

snapshot_path = fetch_artifact(cfg.ingestion.hf_repo, cfg.ingestion.artifacts.snapshot, token=os.environ["HF_TOKEN"])
df = pd.read_parquet(snapshot_path)
print(f"Loaded snapshot rows: {len(df)}  ({cfg.ingestion.artifacts.snapshot})")

main_df, mini_df, eval_df = split.three_way(
    df,
    mini_size=cfg.split.mini_size,
    eval_frac=cfg.split.eval_frac,
    seed=cfg.split.seed,
)
print(f"Splits — main: {len(main_df)}, mini: {len(mini_df)}, eval: {len(eval_df)}")

In [ ]:
# 8 — Embed on GPU
import gc
import torch
from core.text_encoder import encode_all, encode_optional_texts

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")

main_emb = encode_all(list(main_df["composite_text"]), batch_size=64)
mini_emb = encode_all(list(mini_df["composite_text"]), batch_size=64)
eval_emb = encode_all(list(eval_df["composite_text"]), batch_size=64)

main_review_emb = encode_optional_texts(list(main_df["reviews_text"]), batch_size=64)
mini_review_emb = encode_optional_texts(list(mini_df["reviews_text"]), batch_size=64)
eval_review_emb = encode_optional_texts(list(eval_df["reviews_text"]), batch_size=64)

for name, split_df, emb, review_emb in [
    ("main", main_df, main_emb, main_review_emb),
    ("mini", mini_df, mini_emb, mini_review_emb),
    ("eval", eval_df, eval_emb, eval_review_emb),
]:
    assert emb.shape == (len(split_df), cfg.representation.embedding_dim), f"{name}: unexpected shape {emb.shape}"
    assert review_emb.shape == (len(split_df), cfg.representation.embedding_dim), f"{name} review: unexpected shape {review_emb.shape}"
    n_reviews = int((review_emb.sum(axis=1) != 0).sum())
    print(f"{name}: text={emb.shape}  reviews={n_reviews}/{len(split_df)} non-zero")

In [ ]:
# 9 — Embed trailer frames in shards (open_clip ViT-H/14, native 1024-dim)
#
# Shards are persisted to Google Drive so this cell can be re-run after a
# bot-detection interruption — already-completed shards are skipped.
# Set MAX_SHARDS_PER_RUN to a small number to bound a single Colab session
# (None = process everything available in one go).
#
# When a movie's trailer is unavailable or yt-dlp fails, the poster image
# is encoded instead (same CLIP space). Rows with neither trailer nor poster
# remain zero.
from core.trailer_encoder import encode_trailers_sharded

gc.collect()
torch.cuda.empty_cache()

snapshot_stem = Path(cfg.ingestion.artifacts.snapshot).stem
SHARD_SIZE = 50
MAX_SHARDS_PER_RUN = None

results = {}
for name, split_df in [("main", main_df), ("mini", mini_df), ("eval", eval_df)]:
    shard_dir = SHARD_ROOT / snapshot_stem / name
    shard_dir.mkdir(parents=True, exist_ok=True)
    emb, complete = encode_trailers_sharded(
        list(zip(split_df["id"], split_df["trailer_youtube_key"], split_df["poster_path"])),
        shard_dir=shard_dir,
        shard_size=SHARD_SIZE,
        max_shards_per_run=MAX_SHARDS_PER_RUN,
    )
    results[name] = (emb, complete)
    n_nonzero = int((emb.sum(axis=1) != 0).sum())
    print(f"{name}: complete={complete}  ({n_nonzero}/{len(split_df)} embedded so far)")

main_trailer_emb, main_complete = results["main"]
mini_trailer_emb, mini_complete = results["mini"]
eval_trailer_emb, eval_complete = results["eval"]

In [ ]:
# 10 — Gate: refuse to upload until all splits are fully embedded
if not (main_complete and mini_complete and eval_complete):
    raise RuntimeError(
        "Trailer embedding incomplete across one or more splits. "
        "Re-run the previous cell until each split reports complete=True "
        "before publishing to HF."
    )
print("All splits complete — proceeding to upload.")

In [ ]:
# 11 — Upload timestamped artifacts to Hugging Face
from dataset.hub.upload import upload_artifacts

files = upload_artifacts(
    main_df, mini_df, eval_df,
    main_emb, mini_emb, eval_emb,
    main_review_emb=main_review_emb,
    mini_review_emb=mini_review_emb,
    eval_review_emb=eval_review_emb,
    main_trailer_emb=main_trailer_emb,
    mini_trailer_emb=mini_trailer_emb,
    eval_trailer_emb=eval_trailer_emb,
    repo_id=cfg.ingestion.hf_repo,
    artifacts_dir=ARTIFACTS_DIR,
)

print("\nPublished. Paste these into configs/default.yaml under `ingestion.artifacts`:\n")
for split_name, filename in files.items():
    print(f"  {split_name}: {filename}")

## Done

1. Copy the paths printed above into `configs/default.yaml`:

   ```yaml
   ingestion:
     hf_repo: "446f6e6e79/CinePal-embeddings"
     artifacts:
       snapshot: snapshots/snapshot_YYYYMMDD.parquet   # set by `python -m dataset.scraper --upload`
       main: embeddings/main_YYYYMMDD.parquet
       mini: embeddings/mini_YYYYMMDD.parquet
       eval_holdout: embeddings/eval_holdout_YYYYMMDD.parquet
   ```

2. Teammates ingest without a GPU:

   ```bash
   python -m db.ingest             # mini
   python -m db.ingest --set main  # full set
   ```

> **Note on GPU vs CPU floating-point:** embedding outputs may differ by ~1e-6 between
> devices. This is below any meaningful threshold for cosine similarity and can be ignored.

> **Resuming trailer embedding:** if a Colab session ends before all trailer shards are
> complete, re-open the notebook and re-run from cell 1 down to the trailer-shard cell
> (cell 9). Already-completed shards on Google Drive are loaded immediately; only the
> remaining shards are downloaded and embedded.